# 💊 TOÀN TRÌNH BENCHMARK CV THẬT & TỐI ƯU HÓA SIÊU THAM SỐ RAG (COLAB GPU)
### Complete Real CV Benchmark, Parameter Tuning & Scene Evaluation
---
Notebook này được tối ưu để chạy trên **Google Colab (hoặc Kaggle) với GPU T4 / V100 / A100** nhằm:
1. **Chạy suy luận CV thật 100% (YOLOv11-seg + ResNet-18 + PaddleOCR GPU)** trên toàn bộ 101 ảnh trong `pill_images_verified` và 2 ảnh hiện trường `test_1.png`, `test_2.png`.
2. **Thực hiện Grid Search toàn diện (1.944 tổ hợp siêu tham số)** để tìm điểm cân bằng F1, Precision, Recall và các ngưỡng độ tự tin (Confidence Thresholds).
3. **Sử dụng CSDL Dược thư mới nhất từ Kaggle `trannhattruong19691/database-mliotlab`**.
4. **Đo lường chi tiết kết quả trên ảnh đa viên `test_1.png` và `test_2.png`** (số viên nhận diện được, gợi ý Top-3, kiểm tra tương tác thuốc DDI).
5. **Tự động đóng gói và tải về file kết quả JSON** để gửi lại phân tích.

## 🔹 BƯỚC 1: Kiểm Tra GPU, Clone Mã Nguồn & Cài Đặt Môi Trường

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

# 2. Thiết lập thư mục làm việc
import os, shutil, sys

work_dir = '/content' if os.path.exists('/content') else '/kaggle/working'
repo_dir = os.path.join(work_dir, 'repo')

if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

# 3. Clone nhánh mã nguồn mới nhất
!git clone --depth 1 https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git {repo_dir}

%cd {repo_dir}

# 4. Cài đặt các thư viện phụ thuộc
!pip uninstall -y -q paddlepaddle paddlepaddle-gpu paddleocr paddlex
!pip install -q --no-cache-dir -r requirements.txt

# 5. Cài đặt Paddle GPU và PaddleOCR
!pip install -q paddlepaddle-gpu==3.0.0 --index-url https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip install -q paddleocr==3.0.3 paddlex==3.0.3 "numpy==1.26.4" "opencv-python-headless==4.10.0.84" kagglehub

print("\n✅ BƯỚC 1 HOÀN TẤT: Đã cài đặt môi trường GPU và clone repo thành công!")

---
## 🔹 BƯỚC 2: Tự Động Nạp Trọng Số Model AI & CSDL Mới Nhất (`trannhattruong19691/database-mliotlab`)

In [ ]:
import os, shutil, glob
from pathlib import Path
import kagglehub

repo_root = Path(os.getcwd())
seg_dir = repo_root / 'models/segmentation_yolov11_full_finetune'
attr_dir = repo_root / 'models/attribute_resnet18_last_blocks_finetune'
db_seed_dir = repo_root / 'database_seed'

seg_dir.mkdir(parents=True, exist_ok=True)
attr_dir.mkdir(parents=True, exist_ok=True)
db_seed_dir.mkdir(parents=True, exist_ok=True)

print("🚀 Đang tải mô hình AI Deep Learning từ KaggleHub...")

# 1. YOLOv11 Segmentation Weights
print("1/3. Tải YOLOv11 Segmentation...")
try:
    seg_download_path = kagglehub.dataset_download('nnphuchcmus/pill-segmentation-model')
    for f in glob.glob(f'{seg_download_path}/**/*.pt', recursive=True):
        target_pt = seg_dir / 'yolov11m_seg_mediseg_full_finetune_v1.pt'
        shutil.copy(f, target_pt)
        print(f"  ✓ Đã nạp YOLOv11-Seg: {target_pt.name}")
        break
except Exception as e:
    print(f"  ⚠️ Lỗi tải seg hub ({e})")

# 2. ResNet-18 Attribute Weights & Configs
print("2/3. Tải ResNet-18 Attribute...")
try:
    attr_download_path = kagglehub.dataset_download('nnphuchcmus/attrubute-artifact')
    for f in glob.glob(f'{attr_download_path}/**/*', recursive=True):
        if os.path.isfile(f):
            shutil.copy(f, attr_dir / os.path.basename(f))
    print("  ✓ Đã nạp thành công ResNet-18 weights & label mappings!")
except Exception as e:
    print(f"  ⚠️ Lỗi tải attr hub ({e})")

# 3. Database Seeds & SQLite DB (Tải từ dataset mới: trannhattruong19691/database-mliotlab)
print("3/3. Tải CSDL Dược thư mới nhất từ trannhattruong19691/database-mliotlab...")
try:
    db_download_path = kagglehub.dataset_download('trannhattruong19691/database-mliotlab')
    for f in glob.glob(f'{db_download_path}/**/*', recursive=True):
        if f.endswith('.json'):
            shutil.copy(f, db_seed_dir / os.path.basename(f))
        elif f.endswith('.db') or f.endswith('.sqlite'):
            shutil.copy(f, repo_root / 'medication.db')
            print(f"  ✓ Đã nạp file SQLite .db có sẵn: {os.path.basename(f)}")
    print("  ✓ Đã đồng bộ CSDL từ KaggleHub trannhattruong19691/database-mliotlab!")
except Exception as e:
    print(f"  ⚠️ Lỗi tải hub ({e}), đang dùng database_seed có sẵn trong repo.")

# 4. Cấu hình .env và tự động re-seed CSDL SQLite từ database_seed để đảm bảo đủ dữ liệu mới nhất
with open(repo_root / '.env', 'w', encoding='utf-8') as f:
    f.write('DATABASE_URL=sqlite:///./medication.db\n')
    f.write('LLM_PROVIDER=fallback\n')

!python scripts/seed_database.py

print("\n✅ BƯỚC 2 HOÀN TẤT: Toàn bộ model weights và CSDL đã sẵn sàng!")

---
## 🔹 BƯỚC 3: Tự Động Quét & Nạp Tập Ảnh `pill_images_verified`
*(Tự động tìm kiếm mọi file zip hoặc thư mục ảnh bạn đã tải lên Colab)*

In [ ]:
import os, glob, zipfile, shutil
from pathlib import Path
import kagglehub

repo_root = Path(os.getcwd())
verified_target = repo_root / 'pill_images_verified'
verified_target.mkdir(parents=True, exist_ok=True)

print("🔍 Đang tự động quét tìm file zip hoặc thư mục ảnh trên Colab...")

# 1. Quét tìm tất cả các file .zip có khả năng chứa ảnh
search_dirs = ['/content', '/content/data', '/content/drive/MyDrive', '/kaggle/working', '/kaggle/input']
found_zips = []
for sdir in search_dirs:
    if os.path.exists(sdir):
        for zf in glob.glob(f'{sdir}/*.zip'):
            found_zips.append(zf)
        for zf in glob.glob(f'{sdir}/**/*.zip', recursive=True):
            if zf not in found_zips and 'repo' not in zf:
                found_zips.append(zf)

unzipped = False
for z_path in found_zips:
    print(f"📦 Tìm thấy file zip: {z_path}, đang giải nén...")
    try:
        with zipfile.ZipFile(z_path, 'r') as zip_ref:
            zip_ref.extractall(verified_target)
        unzipped = True
        print(f"  ✓ Đã giải nén thành công {os.path.basename(z_path)} vào {verified_target}")
        break
    except Exception as exc:
        print(f"  ⚠️ Lỗi giải nén {z_path}: {exc}")

# 2. Nếu trong thư mục upload trực tiếp đã có sẵn ảnh (ví dụ /content/data hoặc /content/pill_images_verified)
possible_img_sources = [
    Path('/content/pill_images_verified'),
    Path('/content/data/pill_images_verified'),
    Path('/content/data'),
    Path('/kaggle/input/pill-images-verified'),
]
for p_src in possible_img_sources:
    if p_src.exists() and p_src != verified_target:
        # Kiểm tra nếu nguồn này có ảnh thuốc
        imgs_in_src = list(p_src.rglob('*.jpg')) + list(p_src.rglob('*.png'))
        if len(imgs_in_src) >= 10:
            print(f"📁 Tìm thấy {len(imgs_in_src)} ảnh tại {p_src}, đang đồng bộ vào {verified_target}...")
            for item in p_src.iterdir():
                dest = verified_target / item.name
                if item.is_dir():
                    shutil.copytree(item, dest, dirs_exist_ok=True)
                else:
                    shutil.copy2(item, dest)
            unzipped = True
            break

# 3. Tự động xử lý lồng thư mục (nếu zip giải nén thành pill_images_verified/pill_images_verified/...)
nested_subdir = verified_target / 'pill_images_verified'
if nested_subdir.exists() and nested_subdir.is_dir():
    print("🔄 Đang tối ưu cấu trúc thư mục lồng nhau...")
    for item in nested_subdir.iterdir():
        dest = verified_target / item.name
        if not dest.exists():
            shutil.move(str(item), str(dest))

# 4. Kiểm tra tổng kết số lượng ảnh
all_imgs = list(verified_target.rglob('*.jpg')) + list(verified_target.rglob('*.png'))
test_1_found = (verified_target / 'test_1.png').exists() or len(list(verified_target.rglob('test_1.png'))) > 0
test_2_found = (verified_target / 'test_2.png').exists() or len(list(verified_target.rglob('test_2.png'))) > 0

print("=" * 65)
print(f"📊 TỔNG KẾT TẬP DỮ LIỆU ẢNH:")
print(f"  • Tổng số file ảnh tìm thấy: {len(all_imgs)} ảnh.")
print(f"  • test_1.png (Ảnh đa viên 1): {'✓ ĐÃ SẴN SÀNG' if test_1_found else '⚠️ Chưa tìm thấy'}")
print(f"  • test_2.png (Ảnh đa viên 2): {'✓ ĐÃ SẴN SÀNG' if test_2_found else '⚠️ Chưa tìm thấy'}")
print("=" * 65)

if len(all_imgs) >= 100:
    print("🎉 BƯỚC 3 HOÀN TẤT 100%: Tập ảnh verified đã sẵn sàng để benchmark!")
else:
    print(f"⚠️ Hiện tại mới tìm thấy {len(all_imgs)} ảnh. Hãy đảm bảo bạn đã kéo thả file .zip chứa ảnh lên Colab.")

---
## 🔹 BƯỚC 4: Chạy Benchmark Full CV Pipeline Bằng GPU (YOLOv11 + ResNet18 + PaddleOCR)
*(Quét qua toàn bộ ảnh, lưu cache output CV để phục vụ tối ưu siêu tham số tốc độ cao)*

In [ ]:
import sys, os
from pathlib import Path
repo_root = Path(os.getcwd())
sys.path.insert(0, str(repo_root / 'src'))
sys.path.insert(0, str(repo_root))
os.environ['PYTHONPATH'] = f"{str(repo_root / 'src')}:{os.environ.get('PYTHONPATH', '')}"

# Chạy script benchmark toàn diện trên tập verified bằng GPU
!python scripts/benchmark_verified_pills.py --verified-dir pill_images_verified --output-json outputs/benchmark_verified_results.json

print("\n✅ BƯỚC 4 HOÀN TẤT: Đã hoàn thành quét CV thật trên toàn bộ 101 ảnh và lưu kết quả!")

---
## 🔹 BƯỚC 5: Thực Hiện Grid Search Tối Ưu Siêu Tham Số (1.944 Tổ Hợp) & Calibration Độ Tự Tin

In [ ]:
# Chạy Grid Search tìm bộ siêu tham số cân bằng tối ưu nhất
!python scripts/tune_identification_parameters.py --benchmark-file outputs/benchmark_verified_results.json

print("\n✅ BƯỚC 5 HOÀN TẤT: Đã tìm ra các cấu hình siêu tham số tối ưu!")

---
## 🔹 BƯỚC 6: Đo Lường Trực Tiếp Trên 2 Ảnh Hiện Trường `test_1.png` và `test_2.png`

In [ ]:
# Chạy kiểm thử hiện trường trên test_1 và test_2 với mô hình CV thật
!python scripts/evaluate_scenes_direct.py

print("\n✅ BƯỚC 6 HOÀN TẤT: Đã đo lường chi tiết trên test_1.png và test_2.png!")

---
## 🔹 BƯỚC 7: Đóng Gói File Kết Quả Để Gửi Lại Cho AI Agent

In [ ]:
import zipfile
from pathlib import Path

out_zip = Path('/content/benchmark_and_tuning_results.zip') if os.path.exists('/content') else Path('/kaggle/working/benchmark_and_tuning_results.zip')

files_to_pack = [
    Path('outputs/optimized_rag_parameters.json'),
    Path('outputs/benchmark_verified_results.json'),
    Path('outputs/scene_evaluation_results.json'),
]

with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for fp in files_to_pack:
        if fp.exists():
            z.write(fp, arcname=fp.name)
            print(f"  ✓ Đã đóng gói: {fp.name}")

print(f"\n🎉 ĐÃ TẠO FILE KẾT QUẢ: {out_zip}")

# Tự động trigger download trên Google Colab
try:
    from google.colab import files
    files.download(str(out_zip))
    print("⬇️ Đang tải file zip về máy tính của bạn...")
except Exception:
    print(f"👉 Bạn có thể tải file kết quả tại: {out_zip}")